In [0]:
CATALOGO        = "lsdata01"
ESQUEMA         = "DataBricks"
VOL_LANDING     = f"/Volumes/{CATALOGO}/{ESQUEMA}/vol_landing"

In [0]:
display(dbutils.fs.ls("abfss://bronze@lsdata01.dfs.core.windows.net/"))
#abfss://bronze@lsdata01.dfs.core.windows.net/genre.csv

In [0]:
# 1. leemos el archivo csv

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType 

genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
] )

genre_df = spark.read\
    .option("header", True)\
    .schema(genre_schema)\
    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/genre.csv")

genre_df.printSchema()


In [0]:
# Paso 3 - Renombrar Columnas

genre_renamed_df = genre_df\
    .withColumnRenamed("genreId", "genre_id")\
    .withColumnRenamed("genreName", "genre_name")

display(genre_renamed_df)


In [0]:
# Paso 4 - Añadir columnas a una tabla
from pyspark.sql.functions import current_timestamp, lit
genre_final_df = genre_renamed_df\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("produccion"))

display(genre_final_df)
genre_final_df.printSchema()


In [0]:
# Paso 5 - Guardar datos en datalake en formato parket
genre_final_df.write.mode("overwrite").format("parquet").save("abfss://silver@lsdata01.dfs.core.windows.net/genre")


df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/genre")
display(df)
